In [1]:
# Enhanced imports for better training and evaluation
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import copy
from PIL import Image
from tempfile import TemporaryDirectory
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd
from collections import defaultdict

cudnn.benchmark = True
plt.ion()   # interactive mode

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)


In [ ]:
# Check available datasets
!ls


In [ ]:
# Improved data augmentation strategy
# More conservative augmentations to preserve class features
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((256, 256)),  # Consistent resize first
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),  # Less aggressive cropping
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),  # Small rotation for realism
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

# Use the original split dataset first, then can switch to augmented later
data_dir = 'split_dataset'
print(f"Using dataset: {data_dir}")

# Load datasets
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),
                                          data_transforms[x])
                  for x in ['train', 'val', 'test']}

# Improved dataloaders with better batch size
batch_size = 16  # Increased from 4 for better gradient estimates
dataloaders = {
    'train': torch.utils.data.DataLoader(image_datasets['train'], 
                                        batch_size=batch_size,
                                        shuffle=True, 
                                        num_workers=4,
                                        pin_memory=True),
    'val': torch.utils.data.DataLoader(image_datasets['val'], 
                                      batch_size=batch_size,
                                      shuffle=False, 
                                      num_workers=4,
                                      pin_memory=True),
    'test': torch.utils.data.DataLoader(image_datasets['test'], 
                                       batch_size=batch_size,
                                       shuffle=False, 
                                       num_workers=4,
                                       pin_memory=True)
}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}
class_names = image_datasets['train'].classes
num_classes = len(class_names)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Dataset information
print(f"Number of classes: {num_classes}")
print(f"Class names: {class_names}")
print(f"Dataset sizes: {dataset_sizes}")
print(f"Batch size: {batch_size}")


In [ ]:
# Create improved model with better architecture
# Use EfficientNet-V2-S (smaller, often better for limited data)
model_conv = torchvision.models.efficientnet_v2_s(weights='IMAGENET1K_V1')

# Freeze early layers but allow some fine-tuning
# Freeze first 80% of layers, fine-tune the rest
total_params = len(list(model_conv.parameters()))
freeze_until = int(total_params * 0.8)

for i, param in enumerate(model_conv.parameters()):
    if i < freeze_until:
        param.requires_grad = False
    else:
        param.requires_grad = True

# Get the number of input features for the classifier
linear_layer = model_conv.classifier[1]
if isinstance(linear_layer, nn.Linear):
    num_ftrs = linear_layer.in_features
else:
    num_ftrs = 1280  # Default for EfficientNet-V2-S

# Improved classifier with batch normalization and dropout
model_conv.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(num_ftrs, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.3),
    nn.Linear(512, num_classes)
)

model_conv = model_conv.to(device)

# Improved loss function with label smoothing
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Better optimizer settings
# Use different learning rates for different parts
classifier_params = list(model_conv.classifier.parameters())
backbone_params = [p for p in model_conv.parameters() if p.requires_grad and p not in classifier_params]

optimizer_conv = optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5},  # Lower LR for pre-trained layers
    {'params': classifier_params, 'lr': 1e-3}  # Higher LR for new classifier
], weight_decay=0.01)

# Cosine annealing scheduler for better convergence
exp_lr_scheduler = lr_scheduler.CosineAnnealingLR(optimizer_conv, T_max=50, eta_min=1e-6)

print(f"Model moved to {device}")
print(f"Total parameters: {sum(p.numel() for p in model_conv.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model_conv.parameters() if p.requires_grad):,}")


In [ ]:
# Enhanced training function with better logging and early stopping
def train_model(model, criterion, optimizer, scheduler, num_epochs=25, patience=7):
    since = time.time()
    
    # Training history
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            
            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            
            # Store history
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Deep copy the model if it's the best so far
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                epochs_no_improve = 0
            elif phase == 'val':
                epochs_no_improve += 1

        print()
        
        # Early stopping
        if epochs_no_improve >= patience:
            print(f'Early stopping triggered after {epoch+1} epochs')
            break

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')

    # Load best model weights
    model.load_state_dict(best_model_wts)
    return model, history

# Train the model
print("Starting training with improved configuration...")
model_conv, history = train_model(model_conv, criterion, optimizer_conv, 
                                 exp_lr_scheduler, num_epochs=50, patience=10)


In [ ]:
# Comprehensive model evaluation with confusion matrix
def evaluate_model(model, dataloader, class_names, phase='test'):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    # Calculate metrics
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, support = precision_recall_fscore_support(all_labels, all_preds, average=None)
    
    print(f"\\n{phase.upper()} RESULTS:")
    print(f"Overall Accuracy: {accuracy:.4f}")
    print(f"Average Precision: {precision.mean():.4f}")
    print(f"Average Recall: {recall.mean():.4f}")
    print(f"Average F1-Score: {f1.mean():.4f}")
    
    # Create detailed classification report
    print("\\nDetailed Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))
    
    return all_labels, all_preds, all_probs, accuracy

# Evaluate on validation set
val_labels, val_preds, val_probs, val_accuracy = evaluate_model(model_conv, dataloaders['val'], class_names, 'validation')


In [ ]:
# Create and visualize confusion matrix
def plot_confusion_matrix(labels, preds, class_names, title='Confusion Matrix'):
    from sklearn.metrics import confusion_matrix
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    cm = confusion_matrix(labels, preds)
    
    # Calculate percentages
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
    
    # Plot raw counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names, ax=ax1)
    ax1.set_title(f'{title} - Raw Counts')
    ax1.set_xlabel('Predicted Label')
    ax1.set_ylabel('True Label')
    
    # Plot percentages
    sns.heatmap(cm_percent, annot=True, fmt='.1f', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names, ax=ax2)
    ax2.set_title(f'{title} - Percentages')
    ax2.set_xlabel('Predicted Label')
    ax2.set_ylabel('True Label')
    
    plt.tight_layout()
    plt.show()
    
    return cm, cm_percent

# Plot confusion matrix for validation set
cm_val, cm_val_percent = plot_confusion_matrix(val_labels, val_preds, class_names, 'Validation Set')


In [ ]:
# Save the improved model
model_save_path = 'improved_efficientnet_v2_s_vehicle_classifier.pth'
torch.save({
    'model_state_dict': model_conv.state_dict(),
    'optimizer_state_dict': optimizer_conv.state_dict(),
    'class_names': class_names,
    'num_classes': num_classes,
    'val_accuracy': val_accuracy,
    'training_history': history
}, model_save_path)

print(f"Improved model saved as '{model_save_path}'")
print(f"Final validation accuracy: {val_accuracy:.4f}")

# Also save the full model for easy loading
full_model_path = 'improved_efficientnet_v2_s_vehicle_classifier_full.pth'
torch.save(model_conv, full_model_path)
print(f"Full model saved as '{full_model_path}'")
